# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Ignacio Díaz C.
- Nombre de alumno 2: Benjamín Fuentes R.

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.14.3 environment at: C:\Users\ignac\MDS7202\.venv
Audited 8 packages in 29ms


In [11]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [12]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [ ]:
# Escribe aquí tu código

df = load_all_serial(DATA_DIR, 20)

df_opt = df.astype(
    {
        "danceability": "float32",
        "energy": "float32",
        "loudness": "float32",
        "speechiness": "float32",
        "acousticness": "float32",
        "instrumentalness": "float32",
        "liveness": "float32",
        "valence": "float32",
        "tempo": "float32",
        "avg_artist_popularity": "float32",
        "key": "int16",
        "mode": "int16",
        "year": "int32",
        "popularity": "int32",
        "duration_ms": "int32",
        "total_artist_followers": "int32",
    }
)

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?


**Escribe tus respuestas aquí...**

1. Parquet es un formato de almacenamiento binario y columnar. A diferencia de CSV que guarda fila por fila, Parquet guarda columna por columna, lo que permite leer solo las columnas necesarias sin tocar el resto. Además comprime mejor los datos y preserva los tipos sin tener que inferirlos, haciendolo más eficiente para análisis de datos.
2. Apache Arrow es un formato de memoria columnar que sirve como puente entre herramientas. pyarrow es el motor que usa pd.read_parquet internamente, lo que permite leer Parquet directamente a memoria sin conversiones intermedias. A diferencia de pd.read_csv, no necesita inferir tipos ni parsear texto, lo que lo hace más rápido y confiable.
3. float32 usa la mitad de memoria que float64 (4 vs 8 bytes). En ML esto es suficiente porque los modelos toleran pequeñas diferencias de precisión sin que el resultado cambie significativamente.
4. No conviene reducir precisión en cálculos financieros o científicos acumulativos. Los riesgos concretos son overflow (valores que no caben en el rango del tipo), underflow (valores pequeños que se redondean a 0) y acumulación de errores de redondeo.
5. Dos alternativas más eficientes son Polars (escrita en Rust, usa Arrow internamente, más rápida y liviana que pandas) y DuckDB (puede consultar archivos Parquet directamente sin cargar todo en RAM).
6. Se redujo 13 MiB, un 3.6%. Era esperable que fuera poco porque la columna history_text (letras de canciones) domina el uso de memoria, y esa no se puede optimizar con cambios de tipo numérico.
7. float16 tiene muy poca precisión en el rango [0,1], haciendo que muchos valores distintos colapsen al mismo número. Esto degradaría la calidad de las predicciones del modelo al perder resolución justo donde más importa.

In [13]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [14]:
# Escribe aquí tu código
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    with ThreadPoolExecutor() as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)

**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [8]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

**Escribe tus respuestas aquí...**

1. Una operación I/O-bound es aquella donde el cuello de botella es esperar datos externos (disco, red), mientras que CPU-bound es cuando el cuello de botella es el procesador haciendo cálculos. La lectura de archivos desde disco es I/O-bound porque el procesador pasa la mayor parte del tiempo esperando que el disco entregue los datos.
2.  El GIL es un mutex que CPython usa para que solo un thread ejecute código Python a la vez. Existe para proteger la integridad de los objetos en memoria (evitar condiciones de carrera en el manejo de referencias). El problema es que limita el paralelismo real en operaciones CPU-bound con threads, ya que aunque haya varios threads, solo uno corre a la vez.
3. Python se usa porque es simple, legible y tiene un ecosistema enorme. Su rol como "lenguaje de pegamento" es clave: las librerías pesadas como NumPy, Arrow o PyTorch están implementadas en C y liberan el GIL durante sus operaciones, por lo que Python solo las coordina y el trabajo pesado ocurre fuera del GIL.
4. ThreadPoolExecutor conviene para operaciones I/O-bound (lectura de archivos, requests HTTP) porque los threads pueden avanzar mientras otros esperan I/O. Para operaciones puramente CPU-bound se usaría ProcessPoolExecutor, que lanza procesos separados con su propio intérprete y así evita el GIL completamente.
5. Crear el pool tiene un overhead fijo de inicialización de threads. Si los archivos fueran muy pequeños (1 KB), ese overhead sería mayor que el tiempo de leerlos, haciendo que la versión paralela sea más lenta que la serial.
6. Sí, se observó mejora desde el principio. La diferencia empieza a ser notable desde los 4-5 archivos aproximadamente, y se mantiene consistente a lo largo de todo el benchmark, con el paralelo siendo casi el doble de rápido al llegar a los 20 archivos.
7. El speedup no es proporcional al número de threads porque el disco tiene un ancho de banda máximo compartido entre todos los threads, el overhead de crear y coordinar threads consume tiempo, y el pd.concat final es secuencial. Todo esto limita la ganancia real respecto al número teórico de threads disponibles.


# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [15]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [17]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [9]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

**Escribe tus respuestas aquí...**

1. La vectorización en NumPy significa aplicar operaciones sobre arrays completos en vez de elemento por elemento con loops. Internamente NumPy llama a rutinas en C que operan sobre bloques de memoria contiguos, evitando el overhead de interpretar Python en cada iteración.
2. JIT (Just-In-Time) es una técnica que compila el código en tiempo de ejecución, justo antes de usarlo. @numba.njit le indica a Numba que compile la función a código máquina nativo. El modo nopython significa que Numba no puede caer de vuelta al intérprete de Python, si no puede compilar algo, lanza error en vez de ejecutarlo lento.
3. En la primera ejecución Numba compila la función a código máquina, lo cual tarda. Esto se llama warm-up. En el benchmark esto se maneja porque la primera llamada a linear_regression_predict_numba ocurre con solo 10 filas, por lo que el costo de compilación queda aislado en ese punto inicial y no afecta las mediciones posteriores.
4. Polars es una librería de dataframes escrita en Rust, diseñada para ser rápida y eficiente en memoria. Sus principales características son ejecución lazy (optimiza el plan de consulta antes de ejecutar), paralelismo automático y uso de Apache Arrow. Fue diseñada para datasets grandes y ha ganado popularidad porque supera a pandas en velocidad y consumo de memoria.
5. Pandas está escrito en Python/C y ejecuta operaciones de forma eager (inmediata). Polars está escrito en Rust, usa un modelo de ejecución lazy que optimiza las operaciones antes de correrlas, y maneja memoria con Arrow sin hacer copias innecesarias.
6. Pandas tiene overhead de indexación, manejo de tipos nullable y estructuras internas más complejas que NumPy. Aunque internamente usa NumPy para los cálculos, el costo de coordinar Series e índices hace que sea más lento que operar directamente sobre arrays.
7. SIMD son instrucciones del procesador que aplican la misma operación a múltiples datos simultáneamente en un solo ciclo de reloj. NumPy y Polars las aprovechan al almacenar datos en memoria contigua, permitiendo que el procesador opere sobre bloques de 4 u 8 valores a la vez en vez de uno por uno.
8. Numba conviene sobre NumPy cuando la operación tiene loops complejos que no se pueden vectorizar fácilmente, ya que Numba los compila directamente a código máquina. Polars conviene sobre pandas cuando se trabaja con datasets grandes y se hacen múltiples transformaciones encadenadas, donde el optimizador lazy de Polars puede eliminar operaciones innecesarias.
9. La implementación más rápida fue Numba-JIT, con un speedup de hasta 450x respecto a Python puro. Era esperable dado que compila el loop directamente a código máquina nativo y puede aprovechar instrucciones SIMD.
10. Sí hay diferencia, NumPy es consistentemente más rápido que pandas. Esto se debe al overhead de las estructuras internas de pandas (índices, tipos nullable, Series) que añaden costo aunque los cálculos numéricos sean similares.
11. La ventaja empieza a ser evidente desde las primeras filas en escala logarítmica. En términos prácticos, desde 500-1000 filas la diferencia ya es clara y se mantiene creciendo consistentemente.
12. Polars y pandas tuvieron rendimiento muy similar en nuestra medición, con Polars ligeramente mejor. Verificando pd.__version__, si la versión es reciente (2.x), pandas ya incorpora mejoras con Arrow internamente, lo que reduce la brecha con Polars respecto a versiones anteriores.
13. Numba puede superar a NumPy en loops numéricos simples porque compila el loop completo a una sola función nativa optimizada, evitando la creación de arrays intermedios. NumPy en cambio crea arrays temporales en cada operación, lo que implica más accesos a memoria.
14. Si se incluyera el costo de conversión, NumPy y Polars perderían parte de su ventaja, especialmente para volúmenes pequeños donde la conversión tarda más que el cálculo. En producción ese costo no existiría si los datos ya vienen como arrays NumPy (desde sensores o pipelines que generan arrays directamente) o si se usa Polars de principio a fin sin pasar por pandas.
15. Elegiría Numba-JIT por su velocidad superior demostrada en el benchmark. Si hubiera GPU disponible, cambiaría a una implementación con CuPy (NumPy en GPU) o directamente en PyTorch/CUDA, donde la paralelización masiva de la GPU haría el cálculo vectorial sobre 100M filas prácticamente instantáneo.

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [19]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 191.2s | RMSE: 0.1652
n_jobs=-1 → tiempo: 25.6s | RMSE: 0.1652


In [20]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

In [23]:
# Número de procesadores disponibles.
import joblib

joblib.cpu_count()

12

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

**Escribe tus respuestas aquí...**

1. El parámetro n_jobs en los modelos de scikit-learn indica el número de procesos en paralelos que se ralizaran con el modelo, es decir, el número de procesadores a usar en paralelo.
2. Aquí funciona el paralelismo real porque scikit-learn usa joblib por detrás. GIL lo que impide es usar multiples hilos, sin embargo, con joblib en scikit-learn es posible crear procesos independientes que permiten ejecutar varias cosas a la vez, pero de forma predeterminada utilizan memorias independientes, a diferencia de los multihilos.
3. Al utilizar n_jobs=-1, que implica usar todos los procesadores disponibles para procesar el modelo, el tiempo mejoró bastante, llegando a tardar hasta 7 veces menos, de 191.2 a 25.6.
4. En total, se tienen 12 procesadores diponibles, sin embargo, el tiempo disminuó solamente 7.5 veces, lo que no es proporcional al número de procesos utilizados. Esto ocurre porque paralelizar procesos tiene un gasto agregado relacionado con la generación de subprocesos y con la sincronización y el uso de los datos, ya que los procesos usan memoria de forma independiente y no compartida. Por otro lado, siempre existirán procesos que dependerán de otros y que no permiten agilizar más el proceso, por esa razón, no siempre vale la pena paralelizar.
5. No, no hubo diferencia en el RMSE de cada modelo. Esto era esperable porque en scikit-learn gestionan bien y de forma automática la sincronización de los datos, gracias a Joblib, a diferencia de utilizar multiprocessing que es nativa de Python y donde se debe de gestionar manualmente.

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [3]:
%%writefile ./dags/spotify_pipeline_dag.py
# Escribe aquí tu código (copia el template y completa los TODOs)

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/root/airflow/data") 
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────

def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    
    # Se cargan 5 batches de datos en paralelo.
    df = load_all_parallel(DATA_DIR, n_batches = 5)

    # Se guardan los archivos en formato parquet.
    df.to_parquet(OUTPUT_PATH, index = False)

    # Se usa XCom para pasar la ruta del archivo.
    context['ti'].xcom_push(key = 'file_path', value = str(OUTPUT_PATH))


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    
    # Se obtiene la ruta del archivo a través de XCom.
    file_path = context['ti'].xcom_pull(task_ids = 'load_data', key = 'file_path')

    # Se leen los datos formato parquet.
    df = pd.read_parquet(file_path)

    # Se configuran los datos.
    X = df[PARAM_COLS + ["key", "mode", "genre"]]
    y = df["valence"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Se entrena el modelo.
    pipeline = build_pipeline(n_jobs = -1)

    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_time = time.perf_counter() - t0

    print(f"Tiempo total de entrenamiento: {training_time:.2f} [s].")



# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    load_data >> train_model


Writing ./dags/spotify_pipeline_dag.py


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 

```
::group::Log message source details
/root/airflow/logs/dag_id=spotify_pipeline/run_id=manual__2026-06-23T04:30:11.844425+00:00/task_id=load_data/attempt=1.log
::endgroup::
Task Identity ti_id=019ef2be-339a-747b-9441-830d5ff404a9 dag_id=spotify_pipeline task_id=load_data run_id=manual__2026-06-23T04:30:11.844425+00:00 try_number=1 map_index=-1
[2026-06-23T04:30:12.279406Z] INFO - ::group::Pre Execute
[2026-06-23T04:30:16.439134Z] INFO - DAG bundles loaded: dags-folder, example_dags
[2026-06-23T04:30:16.441161Z] INFO - Filling up the DagBag from /root/airflow/dags/spotify_pipeline_dag.py
[2026-06-23T04:30:17.174944Z] WARNING - The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`. category=UserWarning 
[2026-06-23T04:30:17.217416Z] INFO - Task instance is in running state
[2026-06-23T04:30:17.217740Z] INFO -  Previous state of the Task instance: TaskInstanceState.QUEUED
[2026-06-23T04:30:17.217965Z] INFO - Current task name:load_data
[2026-06-23T04:30:17.217871Z] INFO - ::endgroup::
[2026-06-23T04:30:17.218153Z] INFO - Dag name:spotify_pipeline
[2026-06-23T04:30:18.379122Z] INFO - Done. Returned value was: None
[2026-06-23T04:30:18.379426Z] INFO - ::group::Post Execute
[2026-06-23T04:30:18.416400Z] INFO - Task instance in success state
[2026-06-23T04:30:18.416658Z] INFO -  Previous state of the Task instance: TaskInstanceState.RUNNING
[2026-06-23T04:30:18.416866Z] INFO - Task operator:<Task(PythonOperator): load_data>
[2026-06-23T04:30:18.416729Z] INFO - ::endgroup::
```

- Step 2:

```
::group::Log message source details
/root/airflow/logs/dag_id=spotify_pipeline/run_id=manual__2026-06-23T04:30:11.844425+00:00/task_id=train_model/attempt=1.log
::endgroup::
Task Identity ti_id=019ef2be-339b-7316-8895-ae760352a65e dag_id=spotify_pipeline task_id=train_model run_id=manual__2026-06-23T04:30:11.844425+00:00 try_number=1 map_index=-1
[2026-06-23T04:30:18.541390Z] INFO - ::group::Pre Execute
[2026-06-23T04:30:18.563797Z] INFO - DAG bundles loaded: dags-folder, example_dags
[2026-06-23T04:30:18.567058Z] INFO - Filling up the DagBag from /root/airflow/dags/spotify_pipeline_dag.py
[2026-06-23T04:30:19.316146Z] WARNING - The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`. category=UserWarning 
[2026-06-23T04:30:19.338918Z] INFO - Task instance is in running state
[2026-06-23T04:30:19.339202Z] INFO -  Previous state of the Task instance: TaskInstanceState.QUEUED
[2026-06-23T04:30:19.339431Z] INFO - Current task name:train_model
[2026-06-23T04:30:19.339573Z] INFO - Dag name:spotify_pipeline
[2026-06-23T04:30:19.339298Z] INFO - ::endgroup::
[2026-06-23T04:30:25.412700Z] INFO - Tiempo total de entrenamiento: 5.51 [s].
[2026-06-23T04:30:25.446812Z] INFO - Done. Returned value was: None
[2026-06-23T04:30:25.447116Z] INFO - ::group::Post Execute
[2026-06-23T04:30:25.464590Z] INFO - Task instance in success state
[2026-06-23T04:30:25.464839Z] INFO -  Previous state of the Task instance: TaskInstanceState.RUNNING
[2026-06-23T04:30:25.465046Z] INFO - Task operator:<Task(PythonOperator): train_model>
[2026-06-23T04:30:25.465007Z] INFO - ::endgroup::
```

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

1. Un DAG es un grafo utilizado como mapa de un pipeline, que va en una dirección específica (es dirigido) y que no tiene bucles o ciclos (es acíclico). Esto es una propiedad importante para un pipeline porque asegura que el flujo del proceso vaya en una dirección con un inicio y final definido y sin procesos redundantes largos o infinitos.
2. Es una plataforma para programar, orquestar y monitorear pipelines. Está diseñado para gestionar procesos complejos que tienen múltiples dependencias, asegurando que las cosas ocurran en el orden correcto y en el momento requerido. Su unidad mínima de trabajo es el Task, que representa una única acción a ejecutar.
3. Los Operators son componentes que definen una úncia operación dentro de una tarea. El PythonOperator se usa para ejecutar funciones escritas en Python, mientras que el BashOperator ejecuta comandos en la terminal del sistema. El primero se usaría para procesamiento de datos o modelos de Machine Learning, mientras que el segundo para mover archivos o interactuar con el sistema operativo.
4. XCom es un mecanismo de Airflow que permite a las tasks enviarse mensajes de información útil entre si. Internamente, estos mensajes se almacenan en la base de datos de metadatos de Airflow. No es adecuado para pasar DataFrames grandes porque saturaría y colapsaría esta base de datos, la cual está diseñada para guardar estados y configuraciones pequeñas.
5. La alternativa concreta que se usó fue guardar los datos localmente (un archivo Parquet temporal en /tmp/) y pasar únicamente la ruta del archivo a través de XCom. En producción, la alternativa recomendada es guardar esos datos en almacenamiento en la nube, para que cualquier tarea pueda descargarlos independientemente de dónde se esté ejecutando.
6. El parámetro schedule define la frecuencia con la que Airflow ejecutará el DAG de forma automática, utilizando expresiones cron estándar. Para que el pipeline corra todos los días a las 3 AM, el parámetro se configuraría como: schedule="0 3 * * *".
7. Airflow es el orquestador clásico de la industria, mientras que estas otras herramientas son más especializadas y resuelven mejor la construcción de pipelines. La principal crítica a Airflow es su complejidad en su implementación, su alta curva de aprendizaje y su incompatibilidad.
8. Orquestar el pipeline en Airflow permite tener un flujo de trabajo estructurado, eficiente y modular. Esto permite ejecutar el proceso por pasos, facilitando la resolución de errores y la eficiencia de los procesos. Además, en Airflow se puede manejar las ejecuciones, visualizar el historial y los logs aislados por tarea.
9. Si load_data falla, la tarea train_model simplemente no se ejecuta. Airflow no reintenta automáticamente a menos que lo configures. Para controlar esto, le pasas a las tareas o al DAG los parámetros retries (retries=3 para tres intentos).
10. Separar las tareas mejora el debugging porque aísla los fallos. si el entrenamiento falla, los logs de esa tarea te dirán el error exacto sin mezclarse con los logs de otros procesos. A nivel de eficiencia, algún fallo por error de código no genera que tengas que correr todo el código nuevamente, solamente la tarea que falló y sus dependencias, ahorrandote el costo computacional y el tiempo de procesamiento.
11. Podemos generar alertas utilizando los parámetros on_failure_callback y on_success_callback en la configuración del DAG o de las tareas.
12. En un pipeline de producción real, antes de entrenar, agregaría tareas de limpieza de datos para asegurar que no hayan valores nulos o anómalos. Después de entrenar, agregaría una tarea de evaluación para mostrar el rendimiento del modelo actual y finalmente, una tarea que guarde el modelo para su uso futuro.

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>